In [1]:
import os
import sys
import time
import threading
import queue
import numpy as np
import struct
from pynq import allocate
import cv2

# --- 1. 环境初始化 ---
WORK_DIR = '/home/xilinx/jupyter_notebooks/duxu/pynq_vqvae'
sys.path.append('/usr/lib/python3/site-packages')
sys.path.insert(0, '/home/xilinx/jupyter_notebooks/soft/DPU-PYNQ')

from pynq_dpu import DpuOverlay
import vart
import xir

overlay = DpuOverlay(os.path.join(WORK_DIR, 'pl_vq_parallel_260513/dpu.bit'))
vq_ip = overlay.vq_accel_1

In [3]:
enc_out_scale = 0.015625 
dec_in_scale  = 0.03125  
num_vectors = 125 * 175
dim = 64

# Ping-Pong Buffers
vq_in_z_A = allocate(shape=(num_vectors, dim), dtype=np.int8, cacheable=1)
vq_in_z_B = allocate(shape=(num_vectors, dim), dtype=np.int8, cacheable=1)
vq_out_zq_A = allocate(shape=(num_vectors, dim), dtype=np.int8, cacheable=1)
vq_out_zq_B = allocate(shape=(num_vectors, dim), dtype=np.int8, cacheable=1)

vq_in_cb = allocate(shape=(512, 64), dtype=np.float32, cacheable=1)
vq_in_cb[:] = np.load(os.path.join(WORK_DIR, 'codebook.npy')).astype(np.float32)
vq_in_cb.sync_to_device() 

# --- 静态配置 ---
vq_ip.register_map.in_codebook_1 = vq_in_cb.device_address & 0xFFFFFFFF
vq_ip.register_map.in_codebook_2 = vq_in_cb.device_address >> 32

# 转换浮点数为 HEX 发送
vq_ip.register_map.enc_scale = struct.unpack('<I', struct.pack('<f', enc_out_scale))[0]
vq_ip.register_map.dec_scale_inv = struct.unpack('<I', struct.pack('<f', 1.0 / dec_in_scale))[0]
# 【已删除 load_cb_flag 的配置】

def get_dpu_subgraph(path):
    graph = xir.Graph.deserialize(path)
    return graph, graph.get_root_subgraph().toposort_child_subgraph()[1] 

enc_graph, enc_subgraph = get_dpu_subgraph(os.path.join(WORK_DIR, 'xmodel/encoder_zcu111_700x500_old.xmodel'))
enc_runner = vart.Runner.create_runner(enc_subgraph, "run")

dec_graph, dec_subgraph = get_dpu_subgraph(os.path.join(WORK_DIR, 'xmodel/decoder_zcu111_700x500_old.xmodel'))
dec_runner = vart.Runner.create_runner(dec_subgraph, "run")
dec_out_scale = 0.007812 

PRE_DIR = '/home/xilinx/jupyter_notebooks/duxu/pynq_vqvae/imgs_preprocessed'
data_files = sorted([f for f in os.listdir(PRE_DIR) if f.endswith('.npy')])
num_imgs = len(data_files)

dpu_res_queue = queue.Queue(maxsize=2)

def dpu_worker():
    for idx, f in enumerate(data_files):
        t0 = time.perf_counter()
        
        input_data = np.load(os.path.join(PRE_DIR, f))
        in_buf = [np.ascontiguousarray(input_data[np.newaxis])]
        
        # Ping-Pong 分发
        target_vq_buf = vq_in_z_A if idx % 2 == 0 else vq_in_z_B
        out_buf = [np.ndarray((1, 125, 175, 64), dtype=np.int8, buffer=target_vq_buf.data)]
        
        t1 = time.perf_counter()
        job_id = enc_runner.execute_async(in_buf, out_buf)
        enc_runner.wait(job_id)
        t2 = time.perf_counter()
        
        dpu_res_queue.put(('A' if idx % 2 == 0 else 'B', target_vq_buf, t0, t1, t2))
        
    dpu_res_queue.put((None, None, 0, 0, 0))

def run_pipeline():
    t_dpu = threading.Thread(target=dpu_worker)
    t_dpu.start()

    print(f"🚀 Starting Parallel Profiling Pipeline: {num_imgs} images\n")
    all_zq_results = []
    start_time = time.time()

    count = 0
    while True:
        buf_flag, dpu_data_buf, dpu_t0, dpu_t1, dpu_t2 = dpu_res_queue.get()
        if buf_flag is None: 
            break
            
        t_vq_start = time.perf_counter()
        
        # 1. 刷入 DDR
        dpu_data_buf.sync_to_device()  
        t_sync_to = time.perf_counter()
        
        # 2. 绑定对应输出 Buffer
        curr_out_buf = vq_out_zq_A if buf_flag == 'A' else vq_out_zq_B
        vq_ip.mmio.write(0x10, dpu_data_buf.device_address & 0xFFFFFFFF)
        vq_ip.mmio.write(0x14, dpu_data_buf.device_address >> 32)
        vq_ip.mmio.write(0x28, curr_out_buf.device_address & 0xFFFFFFFF)
        vq_ip.mmio.write(0x2C, curr_out_buf.device_address >> 32)
        
        # 3. 启动并解开 ap_ctrl_chain 死锁
        vq_ip.mmio.write(0x00, 0x11) 
        
        # 释放 GIL，极大地改善多线程重叠
        while (vq_ip.mmio.read(0x00) & 0x02) == 0:
            time.sleep(0.001) 
            
        t_vq_done = time.perf_counter()
        
        # 4. 从 DDR 拉取
        curr_out_buf.sync_from_device() 
        t_sync_from = time.perf_counter()
        
        # 保存结果
        zq_snapshot = np.array(curr_out_buf, copy=True)
        t_copy = time.perf_counter()
        all_zq_results.append(zq_snapshot)
        
        # 耗时统计
        dpu_load = (dpu_t1 - dpu_t0) * 1000
        dpu_exec = (dpu_t2 - dpu_t1) * 1000
        vq_sync_in = (t_sync_to - t_vq_start) * 1000
        vq_exec = (t_vq_done - t_sync_to) * 1000
        vq_sync_out = (t_sync_from - t_vq_done) * 1000
        vq_copy = (t_copy - t_sync_from) * 1000
        
        print(f"[Frame {count:02d}] ⏱️ 时耗明细:")
        print(f"  ├─ [DPU Thread] Load NPY: {dpu_load:.2f} ms | Exec: {dpu_exec:.2f} ms")
        print(f"  └─ [VQ  Thread] SyncTo: {vq_sync_in:.2f} ms | VQ Exec: {vq_exec:.2f} ms | SyncFrom: {vq_sync_out:.2f} ms | Copy: {vq_copy:.2f} ms")
        count += 1

    t_dpu.join()
    total_time = (time.time() - start_time) * 1000
    print(f"\n✅ Pipeline Finished. Avg Latency: {total_time/num_imgs:.2f} ms")
    print(f"🚀 Peak FPS: {1000/(total_time/num_imgs):.2f} FPS")

    # --- 保存测试图 ---
    if not os.path.exists('./results'): os.makedirs('./results')
    
    for i in range(len(all_zq_results)):
        z_q_int8 = all_zq_results[i].reshape(1, 125, 175, 64)
        dec_in_buf = [z_q_int8]
        dec_out_buf = [np.empty((1, 500, 700, 3), dtype=np.int8, order='C')]
        job_id = dec_runner.execute_async(dec_in_buf, dec_out_buf)
        dec_runner.wait(job_id)
        
        recon = dec_out_buf[0].astype(np.float32) * dec_out_scale 
        recon = (recon * 0.5 + 0.5).clip(0, 1) 
        recon_img = (recon[0] * 255).astype(np.uint8)
        
        cv2.imwrite(f'./results/recon_{i}.png', cv2.cvtColor(recon_img, cv2.COLOR_RGB2BGR))

if __name__ == "__main__":
    run_pipeline()

🚀 Starting Parallel Profiling Pipeline: 10 images

[Frame 00] ⏱️ 时耗明细:
  ├─ [DPU Thread] Load NPY: 4.44 ms | Exec: 20.53 ms
  └─ [VQ  Thread] SyncTo: 1.91 ms | VQ Exec: 17.44 ms | SyncFrom: 0.28 ms | Copy: 2.17 ms
[Frame 01] ⏱️ 时耗明细:
  ├─ [DPU Thread] Load NPY: 4.00 ms | Exec: 20.99 ms
  └─ [VQ  Thread] SyncTo: 1.82 ms | VQ Exec: 18.43 ms | SyncFrom: 0.29 ms | Copy: 2.35 ms
[Frame 02] ⏱️ 时耗明细:
  ├─ [DPU Thread] Load NPY: 2.87 ms | Exec: 20.56 ms
  └─ [VQ  Thread] SyncTo: 1.82 ms | VQ Exec: 18.46 ms | SyncFrom: 0.30 ms | Copy: 2.33 ms
[Frame 03] ⏱️ 时耗明细:
  ├─ [DPU Thread] Load NPY: 3.10 ms | Exec: 20.47 ms
  └─ [VQ  Thread] SyncTo: 1.81 ms | VQ Exec: 18.46 ms | SyncFrom: 0.29 ms | Copy: 2.22 ms
[Frame 04] ⏱️ 时耗明细:
  ├─ [DPU Thread] Load NPY: 2.76 ms | Exec: 20.56 ms
  └─ [VQ  Thread] SyncTo: 1.81 ms | VQ Exec: 17.45 ms | SyncFrom: 0.30 ms | Copy: 2.35 ms
[Frame 05] ⏱️ 时耗明细:
  ├─ [DPU Thread] Load NPY: 2.90 ms | Exec: 20.48 ms
  └─ [VQ  Thread] SyncTo: 1.83 ms | VQ Exec: 18.45 ms | SyncF

In [4]:
enc_out_scale = 0.015625 
dec_in_scale  = 0.03125  
num_vectors = 125 * 175
dim = 64

# --- 2. Ping-Pong 内存分配 ---
vq_in_z_A = allocate(shape=(num_vectors, dim), dtype=np.int8, cacheable=1)
vq_in_z_B = allocate(shape=(num_vectors, dim), dtype=np.int8, cacheable=1)
vq_out_zq_A = allocate(shape=(num_vectors, dim), dtype=np.int8, cacheable=1)
vq_out_zq_B = allocate(shape=(num_vectors, dim), dtype=np.int8, cacheable=1)

vq_in_cb = allocate(shape=(512, 64), dtype=np.float32, cacheable=1)
vq_in_cb[:] = np.load(os.path.join(WORK_DIR, 'codebook.npy')).astype(np.float32)
vq_in_cb.sync_to_device() 

# 静态配置 VQ 寄存器
vq_ip.register_map.in_codebook_1 = vq_in_cb.device_address & 0xFFFFFFFF
vq_ip.register_map.in_codebook_2 = vq_in_cb.device_address >> 32
vq_ip.register_map.enc_scale = struct.unpack('<I', struct.pack('<f', enc_out_scale))[0]
vq_ip.register_map.dec_scale_inv = struct.unpack('<I', struct.pack('<f', 1.0 / dec_in_scale))[0]

# --- 3. 初始化 DPU Runner ---
def get_dpu_subgraph(path):
    graph = xir.Graph.deserialize(path)
    return graph, graph.get_root_subgraph().toposort_child_subgraph()[1] 

enc_graph, enc_subgraph = get_dpu_subgraph(os.path.join(WORK_DIR, 'xmodel/encoder_zcu111_700x500_old.xmodel'))
enc_runner = vart.Runner.create_runner(enc_subgraph, "run")

dec_graph, dec_subgraph = get_dpu_subgraph(os.path.join(WORK_DIR, 'xmodel/decoder_zcu111_700x500_old.xmodel'))
dec_runner = vart.Runner.create_runner(dec_subgraph, "run")
dec_out_scale = 0.007812 

# =====================================================================
# 【核弹级优化】预计算 Decoder 后处理 LUT 表 (消除 CPU 25ms 计算瓶颈)
# =====================================================================
print("⚙️ Pre-computing Decoder Post-processing LUT...")
post_LUT = np.zeros(256, dtype=np.uint8)
for i in range(256):
    # numpy 在 view为 uint8 时，-128~-1 在内存中表现为 128~255
    val_i8 = i if i < 128 else i - 256 
    val_f = val_i8 * dec_out_scale
    val_f = val_f * 0.5 + 0.5
    post_LUT[i] = int(np.clip(val_f * 255, 0, 255))

# --- 4. 准备数据 ---
PRE_DIR = '/home/xilinx/jupyter_notebooks/duxu/pynq_vqvae/imgs_preprocessed'
data_files = sorted([f for f in os.listdir(PRE_DIR) if f.endswith('.npy')])
num_imgs = len(data_files)

dpu_res_queue = queue.Queue(maxsize=2)

def dpu_worker():
    for idx, f in enumerate(data_files):
        t0 = time.perf_counter()
        
        input_data = np.load(os.path.join(PRE_DIR, f))
        in_buf = [np.ascontiguousarray(input_data[np.newaxis])]
        
        # Ping-Pong 分发
        target_vq_buf = vq_in_z_A if idx % 2 == 0 else vq_in_z_B
        out_buf = [np.ndarray((1, 125, 175, 64), dtype=np.int8, buffer=target_vq_buf.data)]
        
        t1 = time.perf_counter()
        job_id = enc_runner.execute_async(in_buf, out_buf)
        enc_runner.wait(job_id)
        t2 = time.perf_counter()
        
        dpu_res_queue.put(('A' if idx % 2 == 0 else 'B', target_vq_buf, t0, t1, t2))
        
    dpu_res_queue.put((None, None, 0, 0, 0))

def run_pipeline():
    t_dpu = threading.Thread(target=dpu_worker)
    t_dpu.start()

    print(f"🚀 Starting Full Pipeline (Enc + VQ + Dec): {num_imgs} images\n")
    start_time = time.time()
    
    # 缓存前3张图用于最后验证写入
    final_recon_images = []

    count = 0
    while True:
        buf_flag, dpu_data_buf, dpu_t0, dpu_t1, dpu_t2 = dpu_res_queue.get()
        if buf_flag is None: 
            break
            
        t_vq_start = time.perf_counter()
        
        # --- [VQ 阶段] ---
        dpu_data_buf.sync_to_device()  
        t_sync_to = time.perf_counter()
        
        curr_out_buf = vq_out_zq_A if buf_flag == 'A' else vq_out_zq_B
        vq_ip.mmio.write(0x10, dpu_data_buf.device_address & 0xFFFFFFFF)
        vq_ip.mmio.write(0x14, dpu_data_buf.device_address >> 32)
        vq_ip.mmio.write(0x28, curr_out_buf.device_address & 0xFFFFFFFF)
        vq_ip.mmio.write(0x2C, curr_out_buf.device_address >> 32)
        
        # 写入 0x11 解锁 ap_ctrl_chain 死锁
        vq_ip.mmio.write(0x00, 0x11) 
        
        # 睡眠释放 GIL 让后台 DPU 有机会跑
        while (vq_ip.mmio.read(0x00) & 0x02) == 0:
            time.sleep(0.001) 
            
        t_vq_done = time.perf_counter()
        curr_out_buf.sync_from_device() 
        
        # 拷贝数据给 Decoder (防止被后续帧覆盖)
        zq_snapshot = np.array(curr_out_buf, copy=True)
        t_copy_done = time.perf_counter()

        # --- [Decoder 阶段] ---
        z_q_int8 = zq_snapshot.reshape(1, 125, 175, 64)
        dec_in_buf = [z_q_int8]
        dec_out_buf = [np.empty((1, 500, 700, 3), dtype=np.int8, order='C')]
        
        t_dec_start = time.perf_counter()
        job_id = dec_runner.execute_async(dec_in_buf, dec_out_buf)
        dec_runner.wait(job_id)
        t_dec_dpu_done = time.perf_counter()
        
        # ========================================================
        # 极速后处理 (查表法): 把复杂的浮点运算化为一次内存索引映射
        # dec_out_buf[0] 底层是 int8，通过 view(np.uint8) 读出它的 0~255 无符号索引
        # ========================================================
        recon_img = cv2.LUT(dec_out_buf[0].view(np.uint8), post_LUT)
        t_dec_post_done = time.perf_counter()
        
        # 保存前3张验证图
        # if count < 3:
        final_recon_images.append(recon_img[0].copy())
        
        # --- [耗时统计] ---
        dpu_load = (dpu_t1 - dpu_t0) * 1000
        dpu_exec = (dpu_t2 - dpu_t1) * 1000
        vq_sync_in = (t_sync_to - t_vq_start) * 1000
        vq_exec = (t_vq_done - t_sync_to) * 1000
        vq_copy = (t_copy_done - t_vq_done) * 1000
        dec_exec = (t_dec_dpu_done - t_dec_start) * 1000
        
        # 核心见证：查看如今的 Post-process 耗时
        dec_post = (t_dec_post_done - t_dec_dpu_done) * 1000 
        
        print(f"[Frame {count:02d}] ⏱️ 时耗明细:")
        print(f"  ├─ [Enc Thread] Load: {dpu_load:.2f} ms | Enc Exec: {dpu_exec:.2f} ms")
        print(f"  ├─ [VQ  Stage ] Sync: {vq_sync_in:.2f} ms | VQ Exec: {vq_exec:.2f} ms | Copy: {vq_copy:.2f} ms")
        print(f"  └─ [Dec Stage ] Dec Exec: {dec_exec:.2f} ms | Post-process(LUT): {dec_post:.2f} ms")
        # count += 1

    t_dpu.join()
    total_time = (time.time() - start_time) * 1000
    print(f"\n✅ Full Pipeline Finished. Avg Total Latency: {total_time/num_imgs:.2f} ms")
    print(f"🚀 Peak FPS (Enc+VQ+Dec): {1000/(total_time/num_imgs):.2f} FPS")

    # --- 测试完毕后脱机写图验证 ---
    if not os.path.exists('./results'): os.makedirs('./results')
    print("\n🖼️ 正在保存验证图片...")
    for i, img in enumerate(final_recon_images):
        cv2.imwrite(f'./results/recon_{i}.png', cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
        print(f" ✅ 已保存结果至: ./results/recon_{i}.png")

if __name__ == "__main__":
    run_pipeline()
    try: del enc_runner
    except: pass
    try: del dec_runner
    except: pass

⚙️ Pre-computing Decoder Post-processing LUT...
🚀 Starting Full Pipeline (Enc + VQ + Dec): 10 images

[Frame 00] ⏱️ 时耗明细:
  ├─ [Enc Thread] Load: 2.80 ms | Enc Exec: 20.38 ms
  ├─ [VQ  Stage ] Sync: 1.89 ms | VQ Exec: 18.49 ms | Copy: 2.56 ms
  └─ [Dec Stage ] Dec Exec: 19.04 ms | Post-process(LUT): 7.57 ms
[Frame 00] ⏱️ 时耗明细:
  ├─ [Enc Thread] Load: 3.03 ms | Enc Exec: 21.32 ms
  ├─ [VQ  Stage ] Sync: 0.32 ms | VQ Exec: 17.93 ms | Copy: 2.42 ms
  └─ [Dec Stage ] Dec Exec: 25.24 ms | Post-process(LUT): 6.59 ms
[Frame 00] ⏱️ 时耗明细:
  ├─ [Enc Thread] Load: 2.92 ms | Enc Exec: 32.06 ms
  ├─ [VQ  Stage ] Sync: 0.29 ms | VQ Exec: 17.62 ms | Copy: 2.46 ms
  └─ [Dec Stage ] Dec Exec: 28.27 ms | Post-process(LUT): 6.51 ms
[Frame 00] ⏱️ 时耗明细:
  ├─ [Enc Thread] Load: 2.77 ms | Enc Exec: 19.87 ms
  ├─ [VQ  Stage ] Sync: 0.81 ms | VQ Exec: 18.15 ms | Copy: 2.58 ms
  └─ [Dec Stage ] Dec Exec: 17.73 ms | Post-process(LUT): 6.54 ms
[Frame 00] ⏱️ 时耗明细:
  ├─ [Enc Thread] Load: 3.02 ms | Enc Exec: 30.57 